# Содержание
### Строим граф для 1 сбщ
1. Стереть Neo4j базу до нуля (7688)
2. Вызов DeepSeek с промптом (cклеиваем строки в промпт (+ сбщ, + онто))
3. Валидация и исправление ABox JSON c помощью LLM
4. JSON -> Cypher конвертер (LLM возвращает JSON с узлами и нодами, чтобы в конт окно влезало. Cypher не влезал!)
5. Внесение инфы в базу (исполнение всех Cypher команд по очереди)
   

In [ ]:
# 1 стирание базы полное до нуля

In [78]:
!docker stop neo4j_travel || true
!docker rm neo4j_travel || true
!docker volume rm neo4j_travel_data || true

!docker run -d --name neo4j_travel \
  -p 7475:7474 -p 7688:7687 \
  -e NEO4J_AUTH=neo4j/12345678 \
  -v neo4j_travel_data:/data \
  neo4j:5

# !docker ps --filter name=neo4j_travel

neo4j_travel
neo4j_travel
neo4j_travel_data
ebc3ad12b96dbce254ee4cce46b22bb519f60c7201d56aa3e03a221e2d4cf64b


In [51]:
# 2 сборка промпта и запрос к DeepSeek - выход ABox в виде JSON

In [85]:
import os
import json
import requests
import time

PROMPT_PATH = "prompt.txt"
MSG_PATH = "message.txt"
ONTO_PATH = "ontology.txt"

DEEPSEEK_API_KEY = "sk-8da8031877dc4c39ae8d7b624c70f01f"
if not DEEPSEEK_API_KEY:
    raise RuntimeError("Set env var DEEPSEEK_API_KEY first (export DEEPSEEK_API_KEY=...).")

DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
MODEL =  "deepseek-chat" # "deepseek-reasoner"

with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    prompt_template = f.read()

with open(MSG_PATH, "r", encoding="utf-8") as f:
    message_text = f.read().strip()

with open(ONTO_PATH, "r", encoding="utf-8") as f:
    ontology_text = f.read().strip()

prompt = (prompt_template
          .replace("{RDF_ONTOLOGY_HERE}", ontology_text)
          .replace("{TEXT_HERE}", message_text))

url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "Return ONLY JSON. No markdown. Follow the template strictly."},
        {"role": "user", "content": prompt},
    ],
    "temperature": 0.0,
}

t0 = time.perf_counter()
resp = requests.post(url, headers=headers, json=payload, timeout=180)
elapsed = time.perf_counter() - t0
print(f"DeepSeek API latency: {elapsed:.2f}s")
resp.raise_for_status()
data = resp.json()

# DeepSeek API finish reason (transport-level)
api_finish = data["choices"][0].get("finish_reason", "")
content = data["choices"][0]["message"]["content"]

# Save to variable for next cells
answer = content

emoji = "🟢" if api_finish != "length" else "🔴"
print(f"{emoji} API finish_reason = {api_finish}")
print(answer)  # preview


DeepSeek API latency: 84.49s
🟢 API finish_reason = stop
{
  "payload": {
    "ontology_version": "https://example.org/pcfr#",
    "nodes": [
      {
        "label": "Traveller",
        "key": "traveller_1",
        "properties": {
          "pcfr:entityName": "author"
        }
      },
      {
        "label": "Traveller",
        "key": "traveller_2",
        "properties": {
          "pcfr:entityName": "friend"
        }
      },
      {
        "label": "Encounter",
        "key": "encounter_1",
        "properties": {
          "pcfr:encounterDurationSec": 10,
          "pcfr:hasScreeningStatus": "NoExtraProcedures"
        }
      },
      {
        "label": "Encounter",
        "key": "encounter_2",
        "properties": {
          "pcfr:encounterDurationSec": 300,
          "pcfr:hasScreeningStatus": "SecondaryInterrogation"
        }
      },
      {
        "label": "Airport",
        "key": "airport_1",
        "properties": {
          "pcfr:entityName": "ШДГ"
        }


In [84]:
# print(answer)  # preview
print(prompt)

You are an ontology-constrained ABox extractor.

You DO NOT generate Cypher.
You output ONLY JSON.

OUTPUT JSON (STRICT)

Return ONLY valid JSON with this structure:

{
  "payload": {
    "ontology_version": "<string>",
    "nodes": [
      {
        "label": "<ClassName>",
        "key": "<stable_key>",
        "properties": {
          "<datatypeProperty>": <value>
        }
      }
    ],
    "relationships": [
      {
        "from": {"label": "<ClassName>", "key": "<key>"},
        "type": "<ObjectPropertyName>",
        "to": {"label": "<ClassName>", "key": "<key>"}
      }
    ]
  },
  "finish_reason": "🟢 complete | 🟡 partial | 🔴 overflow",
  "reason": "<short explanation why you stopped>",
  "warnings": ["<optional short warnings>"]
}

GLOBAL RULES

- JSON only. No markdown. No code fences. No extra text.
- Use ONLY class names and property names from the ontology (case-sensitive).
- Object properties MUST be in payload.relationships (never as string properties).
- Datatype pro

In [ ]:
# 3 Валидация ABox JSON перед генерацией Cypher

In [77]:

# import json
# import re

# def _strip_code_fences(s: str) -> str:
#     s = s.strip()
#     if s.startswith("```"):
#         s = re.sub(r"^```(?:json)?\s*", "", s)
#         s = re.sub(r"\s*```$", "", s)
#     return s.strip()

# def _one_line(s: str) -> str:
#     return " ".join(str(s).split())

# raw_answer = _strip_code_fences(answer)
# try:
#     abox_obj = json.loads(raw_answer)
#     abox_json = json.dumps(abox_obj, ensure_ascii=False, separators=(",", ":"))
# except Exception:
#     abox_json = _one_line(raw_answer)

# message = message_text
# ontology = ontology_text

# PROMPT_VALID_PATH = "prompt_extr_valid.txt"
# with open(PROMPT_VALID_PATH, "r", encoding="utf-8") as f:
#     prompt_template_valid = f.read()

# prompt_validation = (prompt_template_valid
#     .replace("{MESSAGE}", _one_line(message))
#     .replace("{ONTOLOGY}", _one_line(ontology))
#     .replace("{ABOX_JSON}", abox_json)
# )


# payload_valid = {
#     "model": MODEL,
#     "messages": [
#         {"role": "system", "content": "Return ONLY JSON. No markdown. Output must be a valid ABox JSON and nothing else."},
#         {"role": "user", "content": prompt_validation},
#     ],
#     "temperature": 0.0,
# }

# resp_valid = requests.post(url, headers=headers, json=payload_valid, timeout=180)
# resp_valid.raise_for_status()
# data_valid = resp_valid.json()
# api_finish_valid = data_valid["choices"][0].get("finish_reason", "")
# content_valid = data_valid["choices"][0]["message"]["content"]

# answer_raw = answer
# answer = content_valid

# emoji = "🟢" if api_finish_valid != "length" else "🔴"
# print(f"{emoji} API finish_reason = {api_finish_valid}")
# print(answer)


In [74]:
# print(prompt_validation)
# это ответ валидации но GPT
# answer = """{"payload":{"ontology_version":"https://example.org/pcfr#","nodes":[{"label":"Traveller","key":"traveller_1","properties":{"pcfr:entityName":"author"}},{"label":"Traveller","key":"traveller_2","properties":{"pcfr:entityName":"friend"}},{"label":"Encounter","key":"encounter_1","properties":{"pcfr:encounterDurationSec":10}},{"label":"Encounter","key":"encounter_2","properties":{"pcfr:encounterDurationSec":300}},{"label":"Questioning","key":"questioning_1","properties":{"pcfr:wasAsked":true}},{"label":"Stamping","key":"stamping_1","properties":{}},{"label":"Outcome","key":"outcome_1","properties":{"pcfr:resultCode":"stamped"}},{"label":"Outcome","key":"outcome_2","properties":{"pcfr:resultCode":"stamped"}},{"label":"DocumentCheck","key":"doccheck_1","properties":{"pcfr:presented":true}},{"label":"DocumentCheck","key":"doccheck_2","properties":{"pcfr:presented":true}},{"label":"DocumentCheck","key":"doccheck_3","properties":{"pcfr:presented":true}},{"label":"DocumentCheck","key":"doccheck_4","properties":{"pcfr:presented":true}},{"label":"DocumentCheck","key":"doccheck_5","properties":{"pcfr:presented":true}},{"label":"ReturnTicket","key":"returnticket_1","properties":{}},{"label":"TravelInsurance","key":"travelinsurance_1","properties":{}},{"label":"HotelBookingDoc","key":"hotelbookingdoc_1","properties":{}},{"label":"PaymentCard","key":"paymentcard_1","properties":{}},{"label":"Cash","key":"cash_1","properties":{}},{"label":"ForeignPassport","key":"foreignpassport_1","properties":{}},{"label":"Printout","key":"printout_1","properties":{}},{"label":"PhysicalDocumentRequested","key":"physicaldocumentrequested_1","properties":{}},{"label":"NoExtraProcedures","key":"noextraprocedures_1","properties":{}},{"label":"SecondaryInterrogation","key":"secondaryinterrogation_1","properties":{}},{"label":"City","key":"city_1","properties":{"pcfr:entityName":"Париж"}},{"label":"Country","key":"country_1","properties":{"pcfr:entityName":"France"}},{"label":"Airport","key":"airport_1","properties":{"pcfr:entityName":"ШДГ"}},{"label":"HotelPaymentStatus","key":"hotelpaymentstatus_1","properties":{}},{"label":"CashAmount","key":"cashamount_1","properties":{}},{"label":"HotelPaymentCard","key":"hotelpaymentcard_1","properties":{}},{"label":"VisitPurpose","key":"visitpurpose_1","properties":{}},{"label":"VisitDuration","key":"visitduration_1","properties":{}},{"label":"Employment","key":"employment_1","properties":{}}],"relationships":[{"from":{"label":"Encounter","key":"encounter_1"},"type":"pcfr:hasTraveler","to":{"label":"Traveller","key":"traveller_2"}},{"from":{"label":"Encounter","key":"encounter_1"},"type":"pcfr:hasOutcome","to":{"label":"Outcome","key":"outcome_1"}},{"from":{"label":"Encounter","key":"encounter_1"},"type":"pcfr:atAirport","to":{"label":"Airport","key":"airport_1"}},{"from":{"label":"Encounter","key":"encounter_1"},"type":"pcfr:atCity","to":{"label":"City","key":"city_1"}},{"from":{"label":"Encounter","key":"encounter_1"},"type":"pcfr:atCountry","to":{"label":"Country","key":"country_1"}},{"from":{"label":"Encounter","key":"encounter_1"},"type":"pcfr:hasScreeningStatus","to":{"label":"NoExtraProcedures","key":"noextraprocedures_1"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasTraveler","to":{"label":"Traveller","key":"traveller_1"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasQuestioning","to":{"label":"Questioning","key":"questioning_1"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasStamping","to":{"label":"Stamping","key":"stamping_1"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasOutcome","to":{"label":"Outcome","key":"outcome_2"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:atAirport","to":{"label":"Airport","key":"airport_1"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:atCity","to":{"label":"City","key":"city_1"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:atCountry","to":{"label":"Country","key":"country_1"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasScreeningStatus","to":{"label":"SecondaryInterrogation","key":"secondaryinterrogation_1"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasDocumentCheck","to":{"label":"DocumentCheck","key":"doccheck_1"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasDocumentCheck","to":{"label":"DocumentCheck","key":"doccheck_2"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasDocumentCheck","to":{"label":"DocumentCheck","key":"doccheck_3"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasDocumentCheck","to":{"label":"DocumentCheck","key":"doccheck_4"}},{"from":{"label":"Encounter","key":"encounter_2"},"type":"pcfr:hasDocumentCheck","to":{"label":"DocumentCheck","key":"doccheck_5"}},{"from":{"label":"DocumentCheck","key":"doccheck_1"},"type":"pcfr:requestedDocType","to":{"label":"ReturnTicket","key":"returnticket_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_2"},"type":"pcfr:requestedDocType","to":{"label":"TravelInsurance","key":"travelinsurance_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_3"},"type":"pcfr:requestedDocType","to":{"label":"HotelBookingDoc","key":"hotelbookingdoc_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_4"},"type":"pcfr:requestedDocType","to":{"label":"PaymentCard","key":"paymentcard_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_5"},"type":"pcfr:requestedDocType","to":{"label":"Cash","key":"cash_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_1"},"type":"pcfr:presentedFormat","to":{"label":"Printout","key":"printout_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_2"},"type":"pcfr:presentedFormat","to":{"label":"Printout","key":"printout_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_3"},"type":"pcfr:presentedFormat","to":{"label":"Printout","key":"printout_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_1"},"type":"pcfr:acceptedFormat","to":{"label":"Printout","key":"printout_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_2"},"type":"pcfr:acceptedFormat","to":{"label":"Printout","key":"printout_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_3"},"type":"pcfr:acceptedFormat","to":{"label":"Printout","key":"printout_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_1"},"type":"pcfr:checkMode","to":{"label":"PhysicalDocumentRequested","key":"physicaldocumentrequested_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_2"},"type":"pcfr:checkMode","to":{"label":"PhysicalDocumentRequested","key":"physicaldocumentrequested_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_3"},"type":"pcfr:checkMode","to":{"label":"PhysicalDocumentRequested","key":"physicaldocumentrequested_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_4"},"type":"pcfr:checkMode","to":{"label":"PhysicalDocumentRequested","key":"physicaldocumentrequested_1"}},{"from":{"label":"DocumentCheck","key":"doccheck_5"},"type":"pcfr:checkMode","to":{"label":"PhysicalDocumentRequested","key":"physicaldocumentrequested_1"}},{"from":{"label":"Stamping","key":"stamping_1"},"type":"pcfr:stampsDocument","to":{"label":"ForeignPassport","key":"foreignpassport_1"}},{"from":{"label":"Questioning","key":"questioning_1"},"type":"pcfr:questionTopic","to":{"label":"HotelPaymentStatus","key":"hotelpaymentstatus_1"}},{"from":{"label":"Questioning","key":"questioning_1"},"type":"pcfr:questionTopic","to":{"label":"CashAmount","key":"cashamount_1"}},{"from":{"label":"Questioning","key":"questioning_1"},"type":"pcfr:questionTopic","to":{"label":"HotelPaymentCard","key":"hotelpaymentcard_1"}},{"from":{"label":"Questioning","key":"questioning_1"},"type":"pcfr:questionTopic","to":{"label":"VisitPurpose","key":"visitpurpose_1"}},{"from":{"label":"Questioning","key":"questioning_1"},"type":"pcfr:questionTopic","to":{"label":"VisitDuration","key":"visitduration_1"}},{"from":{"label":"Questioning","key":"questioning_1"},"type":"pcfr:questionTopic","to":{"label":"Employment","key":"employment_1"}},{"from":{"label":"Airport","key":"airport_1"},"type":"pcfr:locatedInCity","to":{"label":"City","key":"city_1"}},{"from":{"label":"City","key":"city_1"},"type":"pcfr:locatedInCountry","to":{"label":"Country","key":"country_1"}}]}}"""

In [53]:
# 4 JSON → Cypher генерация

In [81]:
import json
import re

# ---------- Helpers ----------

def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def neo4j_local_name(name: str) -> str:
    """
    pcfr:entityName -> entityName
    full URI -> last fragment
    """
    if ":" in name:
        return name.split(":")[-1]
    if "#" in name:
        return name.split("#")[-1]
    if "/" in name:
        return name.rsplit("/", 1)[-1]
    return name

def cypher_literal(v):
    if v is None:
        return "null"
    if isinstance(v, bool):
        return "true" if v else "false"
    if isinstance(v, (int, float)):
        return str(v)
    s = str(v).replace("\\", "\\\\").replace("'", "\\'")
    return f"'{s}'"

# ---------- Parse JSON ----------

raw = strip_code_fences(answer)
root = json.loads(raw)

abox = root["payload"]
nodes = abox.get("nodes", [])
rels = abox.get("relationships", [])

# ---------- Generate Constraints ----------

labels = sorted({neo4j_local_name(n["label"]) for n in nodes})

cypher_statements = []

for label in labels:
    cypher_statements.append(
        f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.key IS UNIQUE"
    )

# ---------- Generate Node Statements ----------

for node in nodes:
    label = neo4j_local_name(node["label"])
    key = node["key"]
    props = node.get("properties", {}) or {}

    set_parts = []
    for k, v in props.items():
        prop_key = neo4j_local_name(k)
        set_parts.append(f"n.{prop_key}={cypher_literal(v)}")

    set_clause = ""
    if set_parts:
        set_clause = " SET " + ", ".join(set_parts)

    stmt = f"MERGE (n:{label} {{key:{cypher_literal(key)}}}){set_clause}"
    cypher_statements.append(stmt)

# ---------- Generate Relationship Statements ----------

for rel in rels:
    fl = neo4j_local_name(rel["from"]["label"])
    fk = rel["from"]["key"]
    tl = neo4j_local_name(rel["to"]["label"])
    tk = rel["to"]["key"]
    rt = neo4j_local_name(rel["type"])

    stmt = (
        f"MATCH (a:{fl} {{key:{cypher_literal(fk)}}}), "
        f"(b:{tl} {{key:{cypher_literal(tk)}}}) "
        f"MERGE (a)-[:{rt}]->(b)"
    )

    cypher_statements.append(stmt)

print("LLM finish_reason:", root.get("finish_reason"))
print("LLM reason:", root.get("reason"))
print(f"Prepared {len(cypher_statements)} Cypher statements.")
print("\n--- First 10 statements ---")
for s in cypher_statements[:10]:
    print(s + ";")

LLM finish_reason: 🟢 complete
LLM reason: All entities and relationships extracted from text, including inferred country and evidence format based on allowed enrichments.
Prepared 93 Cypher statements.

--- First 10 statements ---
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Airport) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Cash) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:CashAmount) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:City) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Country) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:DocumentCheck) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Employment) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Encounter) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:ForeignPassport) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:HotelBookingDoc) REQUIRE n.key IS UNIQUE;


In [55]:
# print(cypher_statements)

In [56]:
# 5 исполнение Cypher (каждая строка автономна)

In [82]:
from neo4j import GraphDatabase

URI = "neo4j://localhost:7688"
USER = "neo4j"
PASSWORD = "12345678"
DATABASE = "neo4j"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

try:
    with driver.session(database=DATABASE) as session:
        for i, stmt in enumerate(cypher_statements, 1):
            session.run(stmt).consume()
    print(f"✅ Executed {len(cypher_statements)} statements.")
finally:
    driver.close()

✅ Executed 93 statements.
